In [1]:
from rdflib import Graph, plugin, URIRef, Literal
from rdflib.serializer import Serializer

import re

import json
import requests

import requests_cache

requests_cache.install_cache('plod_cache',backend='memory', expire_after=300)

In [2]:
url = requests.get("https://p-lod.umasscreate.net/api/items?page=1&per_page=100000")
jsonitems = url.text

In [3]:
# print(jsonitems)

In [4]:
g = Graph().parse(data=jsonitems, format='json-ld')

In [5]:
# print(g.serialize(format='turtle').decode('utf-8'))

In [6]:
rdf_construct  = g.query("""
CONSTRUCT { ?new_s a ?type ; ?p_local ?o_local .}
WHERE {?s <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/2> . 

       ?s a ?type_as_item .
       ?type_as_item <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/1> .
       ?type_as_item <http://purl.org/dc/terms/identifier> ?type_as_label .
       BIND(IRI(concat("https://p-lod.umasscreate.net/p-lod/id/",?type_as_label)) as ?type)
       
       ?s <http://purl.org/dc/terms/identifier> ?item_identifier .
       BIND(IRI(CONCAT("https://p-lod.umasscreate.net/p-lod/id/",?item_identifier)) as ?new_s)
       
       ?s ?p_local ?o_local .
       FILTER regex(str(?p_local), "http://p-lod.umasscreate.net/vocabulary#") . 
       
       }
       
""")

In [7]:
p_lod_mapped = g.query("""
SELECT ?new_s ?p WHERE {
?s <http://purl.org/dc/terms/identifier> ?item_identifier .
BIND(IRI(CONCAT("https://p-lod.umasscreate.net/p-lod/id/",?item_identifier)) as ?new_s)

?s <http://omeka.org/s/vocabs/module/mapping#marker> ?p
}
""")

p_lod_mapped = [[i[0],i[1]] for i in p_lod_mapped]

In [8]:
new_g = Graph()
new_g.namespace_manager.bind('p-lod-vocab', URIRef('http://p-lod.umasscreate.net/vocabulary#'))
new_g.namespace_manager.bind('p-lod', URIRef('https://p-lod.umasscreate.net/p-lod/id/'))
for triple in rdf_construct:
    if "https://p-lod.umasscreate.net/api/items/" in triple[2]:
        sub_item_json = requests.get(triple[2]).text
        new_uri = URIRef("https://p-lod.umasscreate.net/p-lod/id/" + json.loads(sub_item_json)['dcterms:identifier'][0]['@value'])
        new_g.add((triple[0],triple[1],new_uri))
    elif str(triple[1]) == "http://p-lod.umasscreate.net/vocabulary#area":
        new_g.add((triple[0],triple[1],Literal(float(triple[2]))))
    else:
        new_g.add(triple)



In [9]:
for m in p_lod_mapped:
    mapping_item_json = requests.get(m[1]).text
    new_geojson = json.loads(mapping_item_json)['o-module-mapping:geojson']
    new_g.add((m[0],
               URIRef("http://p-lod.umasscreate.net/vocabulary#geojson-string"),
               Literal(new_geojson)))

In [10]:
ttl = new_g.serialize(format="turtle").decode('utf-8')

In [11]:
ttl_file = open("p-lod.ttl", "w")
ttl_file.write(ttl)
ttl_file.close()